# 🛠️ Notebook 2: Booking — from broken to correct (bad → best)

We'll build `BookingService` three times:

1. **v1 – naive**: no lock. *Two users can book the same seat.*
2. **v2 – with a lock**: correct under concurrent traffic.
3. **v3 – held seats with expiry**: real systems give you "5 minutes to pay".

Finally we'll talk about how this maps to **database transactions** and **Redis locks** in production.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/movie-ticket-booking
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Re-define the classes (minimal copy from notebook 1)

We keep this notebook runnable on its own — no cross-notebook imports.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
import itertools, threading, time
from typing import Optional

class SeatStatus(Enum):
    FREE = "free"; HELD = "held"; BOOKED = "booked"

@dataclass
class Seat:
    row: str
    number: int
    price: float = 12.5
    status: SeatStatus = SeatStatus.FREE
    held_by: Optional[str] = None
    hold_expires_at: float = 0.0        # unix seconds; 0 = not held
    @property
    def id(self): return f"{self.row}{self.number}"

@dataclass
class Show:
    id: int
    movie: str
    seats: dict                                  # "A1" -> Seat
    _lock: threading.Lock = field(default_factory=threading.Lock)

@dataclass
class Booking:
    id: int
    user: str
    show_id: int
    seats: list
    total: float
    confirmed: bool = False

_booking_id = itertools.count(1)

def make_show(movie="Inception"):
    """Helper: fresh 2x5 show for each demo, so runs are independent."""
    seats = {}
    for row in "AB":
        for n in range(1, 6):
            s = Seat(row, n); seats[s.id] = s
    return Show(1, movie, seats)

print("classes ready")


## 2. v1 — the NAIVE service (buggy on purpose)

Read the code. It looks fine, right? "Check if free, then mark booked." This is called a **check-then-act** pattern, and it is **not atomic**.


In [ ]:
class NaiveBookingService:
    """No lock. Race condition lurking."""
    def book(self, show, user, seat_ids):
        chosen = [show.seats[sid] for sid in seat_ids]

        # --- check ---
        for s in chosen:
            if s.status != SeatStatus.FREE:
                raise RuntimeError(f"{user}: seat {s.id} not free")

        # Imagine the OS pauses our thread here. Another thread could sneak in,
        # see the same seats as FREE, and also "book" them. We force a tiny
        # sleep so the bug is easy to reproduce in a demo.
        time.sleep(0.001)

        # --- act ---
        for s in chosen:
            s.status = SeatStatus.BOOKED
        return Booking(next(_booking_id), user, show.id,
                       chosen, sum(c.price for c in chosen), confirmed=True)


### Watch it fail: 20 users fight for seat A1

In [ ]:
show = make_show()
svc  = NaiveBookingService()
winners, losers = [], []

def try_book(user):
    try:
        svc.book(show, user, ["A1"])
        winners.append(user)
    except Exception:
        losers.append(user)

threads = [threading.Thread(target=try_book, args=(f"u{i}",)) for i in range(20)]
for t in threads: t.start()
for t in threads: t.join()

print(f"winners: {len(winners)} (should be 1!) -> {winners}")
print(f"losers : {len(losers)}")
print(f"seat A1 final status: {show.seats['A1'].status.value}")
if len(winners) > 1:
    print(f"BUG reproduced: {len(winners)} users all 'booked' the SAME seat.")
else:
    print("(got lucky this run -- re-run; with many threads you will usually see >1 winner)")

# Why this reproduces so reliably: time.sleep() RELEASES the GIL, so every thread
# parks inside the critical section and they all wake up believing A1 was free.
# Without the sleep the bug is still there -- it just needs an unlucky preemption,
# which is exactly why races like this survive months of testing and then fire in
# production on a busy Friday night.
assert show.seats["A1"].status is SeatStatus.BOOKED
assert len(winners) + len(losers) == 20, "every thread must finish, win or lose"

**What happened?** Between `check` and `act` the thread got paused. Multiple threads saw the seat as `FREE` and *all* set it to `BOOKED`. Real users would each get a ticket for the same seat → angry phone calls.


## 3. v2 — add a LOCK ✅

A lock makes "check + act" **atomic**: only one thread runs the critical section at a time. Python's `threading.Lock` used with `with` is the simplest way.


In [ ]:
class SafeBookingService:
    """One lock per Show. Check and act happen atomically."""
    def book(self, show, user, seat_ids):
        with show._lock:                                      # the only change
            chosen = [show.seats[sid] for sid in seat_ids]
            for s in chosen:
                if s.status != SeatStatus.FREE:
                    raise RuntimeError(f"{user}: seat {s.id} not free")
            time.sleep(0.001)                                 # same artificial pause
            for s in chosen:
                s.status = SeatStatus.BOOKED
        return Booking(next(_booking_id), user, show.id,
                       chosen, sum(c.price for c in chosen), confirmed=True)

show = make_show()
svc  = SafeBookingService()
winners, losers = [], []

def try_book(user):
    try:
        svc.book(show, user, ["A1"])
        winners.append(user)
    except Exception:
        losers.append(user)

threads = [threading.Thread(target=try_book, args=(f"u{i}",)) for i in range(20)]
for t in threads: t.start()
for t in threads: t.join()

print(f"winners: {winners}  (exactly one)")
print(f"losers : {len(losers)}")
assert len(winners) == 1, "lock did not protect the critical section!"


### Real-world mapping

| In this notebook         | In a real system                                 |
|--------------------------|--------------------------------------------------|
| `threading.Lock`         | **DB row lock** (`SELECT ... FOR UPDATE`) or     |
|                          | **Redis lock** (`SET key NX EX 30`) or           |
|                          | an optimistic UPDATE that checks a version col   |
| One show object in RAM   | One row per seat in a table                      |
| Threads                  | Separate web servers / pods                      |

The **idea** is identical: one actor at a time inside the critical section.


## 4. v3 — HOLD seats before paying ⏱️

Real booking sites don't book immediately — they **hold** your seats for ~5 minutes while you enter your card. If you don't pay, the hold expires and the seat goes back to FREE.

We'll add:

- `hold(user, seat_ids, ttl)` → seats become `HELD`.
- `confirm(booking)` → `HELD` → `BOOKED` after "payment".
- An expiry check that releases stale holds.


In [ ]:
HOLD_TTL = 0.5    # seconds (tiny so the demo is fast; real life ~= 300s)

class HoldingBookingService:
    """Adds a timed hold step (like real ticket sites)."""

    def _expire_stale(self, show):
        """Release HELD seats whose timer has run out."""
        now = time.time()
        for s in show.seats.values():
            if s.status == SeatStatus.HELD and s.hold_expires_at < now:
                s.status = SeatStatus.FREE
                s.held_by = None
                s.hold_expires_at = 0.0

    def hold(self, show, user, seat_ids, ttl=HOLD_TTL):
        with show._lock:
            self._expire_stale(show)
            chosen = [show.seats[sid] for sid in seat_ids]
            if any(s.status != SeatStatus.FREE for s in chosen):
                raise RuntimeError(f"{user}: seat not free")
            for s in chosen:
                s.status = SeatStatus.HELD
                s.held_by = user
                s.hold_expires_at = time.time() + ttl
        return Booking(next(_booking_id), user, show.id,
                       chosen, sum(c.price for c in chosen), confirmed=False)

    def confirm(self, booking, show):
        """Called after payment succeeds."""
        with show._lock:
            for s in booking.seats:
                if s.status != SeatStatus.HELD or s.held_by != booking.user:
                    raise RuntimeError("hold expired or stolen -- please retry")
            for s in booking.seats:
                s.status = SeatStatus.BOOKED
                s.held_by = None
                s.hold_expires_at = 0.0
            booking.confirmed = True
        return booking

    def cancel(self, booking, show):
        with show._lock:
            for s in booking.seats:
                if s.held_by == booking.user and s.status == SeatStatus.HELD:
                    s.status = SeatStatus.FREE
                    s.held_by = None
                    s.hold_expires_at = 0.0


### Scenario A: happy path — hold, pay, confirm

In [ ]:
show = make_show()
svc  = HoldingBookingService()

b = svc.hold(show, "ada", ["A1", "A2"])
print("held:", [s.id for s in b.seats], "status:", show.seats["A1"].status.value)

# ... user pays with Stripe here ...
svc.confirm(b, show)
print("confirmed:", b.confirmed, "status now:", show.seats["A1"].status.value)


### Scenario B: user walks away — hold expires, seat returns to FREE

In [ ]:
show = make_show()
svc  = HoldingBookingService()

svc.hold(show, "distracted_bob", ["B3"], ttl=0.1)
print("just after hold:", show.seats["B3"].status.value)

time.sleep(0.15)   # bob goes for coffee

# Any next call triggers _expire_stale
svc.hold(show, "grace", ["B3"])
print("grace grabbed B3! status:", show.seats["B3"].status.value,
      "held_by:", show.seats["B3"].held_by)


### Scenario C: cancel a hold manually

In [ ]:
show = make_show()
svc  = HoldingBookingService()

b = svc.hold(show, "carol", ["A5"])
print("before cancel:", show.seats["A5"].status.value)
svc.cancel(b, show)
print("after cancel :", show.seats["A5"].status.value)


## 4b. Verify the design — the invariants, under load

The scenarios above each show one happy path. The property that actually matters is the one a
booking system can never violate:

> **A seat is BOOKED by at most one booking, no matter how many people click at once.**

Below we assert exactly that, plus the state-machine rules around holds. The concurrency test is
the important one — it is the same test that **fails** against `NaiveBookingService`, which is how
we know the lock is doing real work and not just decorating the code.

In [ ]:
# --- 1. Under contention, exactly one booking wins each seat ---------------
def hammer(service, seat_ids, n_users=30, ttl=None):
    """Fire n_users threads at the same seats; return (winners, errors)."""
    s = make_show()
    won, err = [], []
    def attempt(u):
        try:
            if ttl is None:
                service.book(s, u, seat_ids)
            else:
                service.hold(s, u, seat_ids, ttl=ttl)
            won.append(u)
        except Exception:
            err.append(u)
    ts = [threading.Thread(target=attempt, args=(f"u{i}",)) for i in range(n_users)]
    for t in ts: t.start()
    for t in ts: t.join()
    return s, won, err

s, won, err = hammer(SafeBookingService(), ["A1"])
assert len(won) == 1,                    f"exactly one winner, got {len(won)}"
assert len(err) == 29,                   "everyone else must be told no"
assert s.seats["A1"].status is SeatStatus.BOOKED

# The SAME test against the naive service is what fails — proof the lock matters.
_, naive_won, _ = hammer(NaiveBookingService(), ["A1"])
assert len(naive_won) > 1, ("the naive service should double-book; if this ever passes, the "
                            "demo has stopped demonstrating anything")

# --- 2. Multi-seat requests are all-or-nothing ----------------------------
s, won, err = hammer(HoldingBookingService(), ["A1", "A2", "A3"], ttl=5)
assert len(won) == 1, "a 3-seat request must not be split between two users"
assert all(s.seats[sid].held_by == won[0] for sid in ("A1", "A2", "A3")), \
    "the winner holds every seat it asked for"
assert s.seats["A4"].status is SeatStatus.FREE, "untouched seats stay free"

# Partial availability fails the WHOLE batch — no half-booked orders.
svc = HoldingBookingService()
s2 = make_show()
svc.hold(s2, "first", ["B1"], ttl=5)
try:
    svc.hold(s2, "second", ["B1", "B2"], ttl=5)
    raise AssertionError("a batch containing a taken seat must be rejected")
except RuntimeError:
    pass
assert s2.seats["B2"].status is SeatStatus.FREE, "the failed batch left NO partial hold"

# --- 3. The hold state machine ---------------------------------------------
svc, s3 = HoldingBookingService(), make_show()
b = svc.hold(s3, "ada", ["A1"], ttl=5)
assert s3.seats["A1"].status is SeatStatus.HELD and not b.confirmed

# You cannot confirm someone else's hold.
stolen = Booking(999, "mallory", s3.id, b.seats, b.total)
try:
    svc.confirm(stolen, s3)
    raise AssertionError("confirming another user's hold must be rejected")
except RuntimeError:
    pass

svc.confirm(b, s3)
assert b.confirmed and s3.seats["A1"].status is SeatStatus.BOOKED
assert s3.seats["A1"].held_by is None, "confirming clears the hold metadata"

# A BOOKED seat is terminal: no hold can reclaim it.
try:
    svc.hold(s3, "grace", ["A1"], ttl=5)
    raise AssertionError("a booked seat must never be re-held")
except RuntimeError:
    pass

# --- 4. Expiry releases the seat; confirming an expired hold fails ---------
svc, s4 = HoldingBookingService(), make_show()
stale = svc.hold(s4, "bob", ["B5"], ttl=0.05)
time.sleep(0.1)
svc._expire_stale(s4)
seat_id = stale.seats[0].id
assert s4.seats[seat_id].status is SeatStatus.FREE, "an expired hold releases the seat"
assert s4.seats[seat_id].held_by is None
try:
    svc.confirm(stale, s4)
    raise AssertionError("an expired hold must not be confirmable")
except RuntimeError:
    pass

# --- 5. Money is derived from the seats, never stored independently -------
svc, s5 = HoldingBookingService(), make_show()
order = svc.hold(s5, "zoe", ["A1", "A2"], ttl=5)
assert order.total == sum(seat.price for seat in order.seats) == 25.0

print("all booking invariants hold ✅  (including the one the naive service fails)")

## 5. Edge cases the current design still has

Good OOD exercises stop here and ask: *"what did we NOT handle?"*

- **Partial failure**: user asks for A1 & A2; A1 free, A2 already held. Currently we raise on the whole batch — usually the right call, but make it explicit.
- **One giant lock per show** can become a bottleneck on a very popular show. In production people use **per-seat row locks** in the DB instead.
- **Payment failure** should trigger `cancel` (put seat back to FREE). We left the integration as a TODO.
- **Clock skew** across servers breaks `hold_expires_at`. Real systems store the expiry in the **database**, so every server sees the same clock.
- **Double-click protection**: the *same* user retrying shouldn't accidentally book two sets of seats. Use an **idempotency key**.

Try extending the code above — e.g. add an `idempotency_key` parameter to `hold()` that remembers the first call's result.


## 6. Takeaways

- **Split responsibilities**: Seat / Show / Booking / Payment each do one thing.
- **Check-then-act is a bug** without a lock — even in a single process.
- **A lock turns a check-then-act into an atomic operation.**
- **Real systems use hold + expiry** so abandoned carts don't freeze seats forever.
- The same pattern (`threading.Lock` → DB `SELECT FOR UPDATE` → Redis `SET NX EX`) shows up in *every* booking, ride-hail, and inventory problem.

Next lab suggestion: **distributed locks with Redis** — same pattern, across many servers.
